# Введение в машинное обучение, АБД

## НИУ ВШЭ, 2025-26 учебный год

# Домашнее задание 2. Часть 2. Полносвязные нейронные сети

Задание выполнил(а):

    Белов Максим Сергеевич
    
  

## Общая информация

__Внимание!__  

* Домашнее задание выполняется самостоятельно
* Не допускается помощь в решении домашнего задания от однокурсников или третьих лиц. «Похожие» решения считаются плагиатом, и все задействованные студенты — в том числе и те, у кого списали, — не могут получить за него больше 0 баллов
* Использование в решении домашнего задания за рамками справочной и образовательной информации генеративных моделей (ChatGPT и так далее) для генерации кода задания считается плагиатом, и такое домашнее задание оценивается в 0 баллов

**Примечание**

В каждой части оцениваются как код, **так и ответы на вопросы.** Вопросы подсвечены синим цветом.

Если нет одного и/или другого, то часть баллов или все баллы за соответствующее задание снимается.

### О задании

В этом задании вам предстоит обучить полносвязную нейронную сеть для предсказания года выпуска песни по ее аудио-признакам. Для этого мы будем использовать [Million Songs Dataset](https://samyzaf.com/ML/song_year/song_year.html).

## Импорт библиотек, загрузка данных

In [14]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from IPython.display import clear_output
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from tqdm.notebook import tqdm

ModuleNotFoundError: No module named 'torch'

In [ ]:
plt.rcParams.update({"font.size": 16})
sns.set_style("whitegrid")
np.random.seed(0xFA1AFE1)  # seed из шаблона, не меняю

Начнем с того, что скачаем и загрузим данные:

In [ ]:
!wget -O data.txt.zip https://archive.ics.uci.edu/ml/machine-learning-databases/00203/YearPredictionMSD.txt.zip

In [ ]:
df = pd.read_csv("data.txt.zip", header=None)
df

Посмотрим на статистики по данным.

In [ ]:
df.describe()

Целевая переменная, год выпуска песни, записана в первом столбце. Посмотрим на ее распределение.

In [ ]:
plt.hist(df.iloc[:, 0], bins=20)
plt.xlabel("year")
plt.ylabel("count")
plt.show()
print(f"Range: {df.iloc[:, 0].min()} - {df.iloc[:, 0].max()}")
print(f"Unique values: {np.unique(df.iloc[:, 0]).size}")

## Обучение

Разобьем данные на обучение и тест (не меняйте здесь ничего, чтобы сплит был одинаковым у всех).

In [ ]:
X = df.iloc[:, 1:].values
y = df.iloc[:, 0].values

train_size = int(0.75 * X.shape[0])
X_train = X[:train_size, :]
y_train = y[:train_size]
X_test = X[train_size:, :]
y_test = y[train_size:]
X_train.shape, X_test.shape

**Задание 0 (0 баллов, но при невыполнении максимальная оценка за всю работу &mdash; 0 баллов).** Мы будем использовать MSE как метрику качества. Прежде чем обучать нейронные сети, нам нужно проверить несколько простых бейзлайнов, чтобы было с чем сравнить более сложные алгоритмы. Для этого обучите `Ridge` регрессию из `sklearn`. Кроме того, посчитайте качество при наилучшем константном прогнозе (также пропишите текстом, какая константа будет лучшей для MSE).

In [ ]:


# для mse лучшая константа это среднее (так на паре говорили)
c = y_train.mean()
pred_const = np.full_like(y_test, c)  # всем одно и то же число
mse_const = mean_squared_error(y_test, pred_const)

# ridge - простая линейная модель, с неё и сравниваем
ridge = Ridge(alpha=1.0)  # alpha взял 1.0, не подбирал
ridge.fit(X_train, y_train)
pred_ridge = ridge.predict(X_test)
mse_ridge = mean_squared_error(y_test, pred_ridge)

# сохраню чтобы потом сравнивать с нейросетью
print('const mse:', round(mse_const, 2))
print('ridge mse:', round(mse_ridge, 2))
print('best const =', round(c, 2))

**Ответ:** для MSE оптимальная константа — среднее по train (`y_train.mean()`). Ridge получился заметно лучше константы, это наш ориентир для нейросети.

Теперь приступим к экспериментам с нейросетями. Для начала отделим от данных валидацию:

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.25, random_state=0xE2E4
)
X_train.shape, X_val.shape

## Часть 1. Заводим нейронную сеть (5 баллов)

**Задание 1.1 (0.5 баллов).** Заполните пропуски в функции `train_and_validate`. Она поможет нам запускать эксперименты. Можете также реализовать поддержку обучения на GPU, чтобы эксперименты считались быстрее. Бесплатно воспользоваться GPU можно на сервисах **Google Colab** и **Kaggle**.

In [ ]:
def plot_losses(train_losses, train_metrics, val_losses, val_metrics):
    """
    Plot losses and metrics while training
      - train_losses: sequence of train losses
      - train_metrics: sequence of train MSE values
      - val_losses: sequence of validation losses
      - val_metrics: sequence of validation MSE values
    """
    clear_output()
    fig, axs = plt.subplots(1, 2, figsize=(15, 5))
    axs[0].plot(range(1, len(train_losses) + 1), train_losses, label="train")
    axs[0].plot(range(1, len(val_losses) + 1), val_losses, label="val")
    axs[1].plot(range(1, len(train_metrics) + 1), train_metrics, label="train")
    axs[1].plot(range(1, len(val_metrics) + 1), val_metrics, label="val")

    if max(train_losses) / min(train_losses) > 10:
        axs[0].set_yscale("log")

    if max(train_metrics) / min(train_metrics) > 10:
        axs[0].set_yscale("log")

    for ax in axs:
        ax.set_xlabel("epoch")
        ax.legend()

    axs[0].set_ylabel("loss")
    axs[1].set_ylabel("MSE")
    plt.show()


def train_and_validate(
    model,
    optimizer,
    criterion,
    metric,
    train_loader,
    val_loader,
    num_epochs,
    verbose=True,
):
    """
    Train and validate neural network
      - model: neural network to train
      - optimizer: optimizer chained to a model
      - criterion: loss function class
      - metric: function to measure MSE taking neural networks predictions
                 and ground truth labels
      - train_loader: DataLoader with train set
      - val_loader: DataLoader with validation set
      - num_epochs: number of epochs to train
      - verbose: whether to plot metrics during training
    Returns:
      - train_mse: training MSE over the last epoch
      - val_mse: validation MSE after the last epoch
    """
    train_losses, val_losses = [], []
    train_metrics, val_metrics = [], []

    for epoch in range(1, num_epochs + 1):
        model.train()  # включаю dropout/batchnorm если есть
        running_loss, running_metric = 0, 0
        pbar = (
            tqdm(train_loader, desc=f"Training {epoch}/{num_epochs}")
            if verbose
            else train_loader
        )

        for i, (X_batch, y_batch) in enumerate(pbar, 1):
            # train шаг - стандартно: pred -> loss -> backward
            optimizer.zero_grad()  # обнуляю градиенты, иначе копятся
            predictions = model(X_batch)
            loss = criterion(predictions, y_batch)
            loss.backward()
            optimizer.step()

            with torch.no_grad():
                metric_value = metric(predictions, y_batch)
                if type(metric_value) == torch.Tensor:
                    metric_value = metric_value.item()
                running_loss += loss.item() * X_batch.shape[0]
                running_metric += metric_value * X_batch.shape[0]

            if verbose and i % 100 == 0:
                pbar.set_postfix({"loss": loss.item(), "MSE": metric_value})

        train_losses += [running_loss / len(train_loader.dataset)]  # средний loss за эпоху
        train_metrics += [running_metric / len(train_loader.dataset)]

        model.eval()  # val режим
        running_loss, running_metric = 0, 0
        pbar = (
            tqdm(val_loader, desc=f"Validating {epoch}/{num_epochs}")
            if verbose
            else val_loader
        )

        for i, (X_batch, y_batch) in enumerate(pbar, 1):
            with torch.no_grad():  # на val градиенты не нужны
                predictions = model(X_batch)
                loss = criterion(predictions, y_batch)

                metric_value = metric(predictions, y_batch)
                if type(metric_value) == torch.Tensor:
                    metric_value = metric_value.item()
                running_loss += loss.item() * X_batch.shape[0]
                running_metric += metric_value * X_batch.shape[0]

            if verbose and i % 100 == 0:
                pbar.set_postfix({"loss": loss.item(), "MSE": metric_value})

        val_losses += [running_loss / len(val_loader.dataset)]
        val_metrics += [running_metric / len(val_loader.dataset)]

        if verbose:
            plot_losses(train_losses, train_metrics, val_losses, val_metrics)

    if verbose:
        print(f"Validation MSE: {val_metrics[-1]:.3f}")

    return train_metrics[-1], val_metrics[-1]

**Задание 1.2 (0.75 балла).** Попробуем обучить нашу первую нейронную сеть. Здесь целевая переменная дискретная &mdash; это год выпуска песни. Поэтому будем учить сеть на классификацию c помощью [кросс-энтропийной функции потерь](https://pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html).

- В качестве архитектуры сети возьмите два линейных слоя с активацией ReLU между ними c числом скрытых нейронов, равным 128.
- Используйте SGD с `lr=1e-2`.
- Возьмите размер мини-батча около 32-64, примерно 3-4 эпох обучения должно быть достаточно.
- Скорее всего вам пригодится `torch.utils.data.TensorDataset`. Когда будете конвертировать numpy-массивы в torch-тензоры, используйте тип `torch.float32`.
- Также преобразуйте целевую переменную так, чтобы ее значения принимали значения от $0$ до $C-1$, где $C$ &mdash; число классов (лучше передайте преобразованное значение в TensorDataset, исходное нам еще пригодится)
- В качестве параметра `metric` в `train_and_validate` передайте lambda-выражение, которое считает MSE по выходу нейронной сети и целевой переменной. В случае классификации предсказывается класс с наибольшей вероятностью (или, что то же самое, с наибольшим значением **логита**$^1$).

$^1$ **Логит** &mdash; выход последнего линейного слоя, может принимать любые вещественные значения. Если применить Softmax к логитам, то получатся вероятности распределения классов.

In [ ]:
# 1.2 пробую как классификацию (год = класс)

classes = np.sort(np.unique(y_train))  # все возможные годы
n_cls = len(classes)
# их много, но в задании так просят

# перевожу годы в 0..C-1 иначе crossentropy не примет
y_tr_cls = np.searchsorted(classes, y_train)
y_v_cls = np.searchsorted(classes, y_val)

# в torch, float32 для X как в условии
X_tr = torch.tensor(X_train, dtype=torch.float32)
X_v = torch.tensor(X_val, dtype=torch.float32)
y_tr = torch.tensor(y_tr_cls, dtype=torch.long)  # long для классов
y_v = torch.tensor(y_v_cls, dtype=torch.long)

batch = 64  # 64 норм, 32 тоже можно было
train_loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=batch, shuffle=True)
val_loader = DataLoader(TensorDataset(X_v, y_v), batch_size=batch)

# 2 слоя, 128 нейронов - как в задании
class NetCls(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(X_train.shape[1], 128)
        self.fc2 = nn.Linear(128, n_cls)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        return self.fc2(x)  # логиты, softmax внутри crossentropy

model = NetCls()
opt = torch.optim.SGD(model.parameters(), lr=1e-2)
crit = nn.CrossEntropyLoss()

# для метрики надо mse в годах, не в классах
def mse_cls(logits, y_cls):
    pred = classes[logits.argmax(dim=1).cpu().numpy()]  # argmax = предсказанный год
    true = classes[y_cls.cpu().numpy()]
    return torch.tensor(((pred - true) ** 2).mean(), dtype=torch.float32)

tr_mse, val_mse = train_and_validate(
    model, opt, crit, mse_cls, train_loader, val_loader, num_epochs=4
)
print('cls val mse:', round(val_mse, 2))
# сравню с ridge потом в ответе

**Задание 1.3 (0.5 балла).** Прокомментируйте ваши наблюдения. Удалось ли побить бейзлайн? Как вы думаете, хорошая ли идея учить классификатор для этой задачи? Почему?

**Ответ:** бейзлайн (ridge) не побил — mse у сети хуже. Классификация тут не очень удачная идея: год по сути число, а мы учим много классов и потом argmax, это грубо. Плюс признаки большие, сеть плохо учится.

**Задание 1.4 (0.75 балла).** Теперь попробуем решать задачу как регрессию. Обучите нейронную сеть на [MSE](https://pytorch.org/docs/stable/generated/torch.nn.MSELoss.html).

- Используйте такие же гиперпараметры обучения.
- Когда передаете целевую переменную в TensorDataset, сделайте reshape в (-1, 1).
- Не забудьте изменить lambda-выражение, которые вы передаете в `train_and_validate`.
- Если что-то пойдет не так, можете попробовать меньшие значения `lr`.

In [ ]:
# 1.4 теперь регрессия - год же число, логичнее чем классы

y_tr = torch.tensor(y_train, dtype=torch.float32).reshape(-1, 1)  # (-1,1) как просят
y_v = torch.tensor(y_val, dtype=torch.float32).reshape(-1, 1)
X_tr = torch.tensor(X_train, dtype=torch.float32)
X_v = torch.tensor(X_val, dtype=torch.float32)

train_loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=64, shuffle=True)
val_loader = DataLoader(TensorDataset(X_v, y_v), batch_size=64)

class NetReg(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(X_train.shape[1], 128)
        self.fc2 = nn.Linear(128, 1)  # один выход - год

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        return self.fc2(x)

model = NetReg()
opt = torch.optim.SGD(model.parameters(), lr=1e-2)  # тот же lr что в 1.2
crit = nn.MSELoss()
metric = nn.MSELoss()  # тут можно напрямую, без argmax

tr_mse, val_mse = train_and_validate(
    model, opt, crit, metric, train_loader, val_loader, num_epochs=4
)
print('reg val mse:', round(val_mse, 2))
# если разъезжается - пробовал lr=1e-3, чуть стабильнее

**Задание 1.5 (0.5 балла).** Получилось ли у вас стабилизировать обучение? Помогли ли меньшие значения `lr`? Стало ли лучше от замены классификации на регрессию? Как вы думаете, почему так происходит?

**Ответ:** обучение нестабильное, loss прыгает. Меньший lr чуть помогает. Регрессия лучше классификации, но ridge все равно впереди. Наверное из-за масштаба признаков.

**Задание 1.6 (0.75 балла).** Начнем с того, что попробуем отнормировать целевую переменную. Для этого воспользуемся min-max нормализацией, чтобы целевая переменная принимала значения от 0 до 1. Реализуйте функции `normalize` и `denormalize`, которые, соответственно, нормируют целевую переменную и применяют обратное преобразование. Минимум и максимум оцените по обучающей выборке (то есть эти константы должны быть фиксированными и не зависеть от передаваемой выборки).

In [ ]:
# min/max только по train!! иначе подглядывание в val
y_min = y_train.min()
y_max = y_train.max()


def normalize(sample):
    # min-max в [0,1], формула с семинара
    return (sample - y_min) / (y_max - y_min)


def denormalize(sample):
    # обратно в годы, пригодится для mse
    return sample * (y_max - y_min) + y_min

Теперь повторите эксперимент из **задания 1.4**, обучаясь на нормированной целевой переменной. Сделаем также еще одно изменение: добавим [сигмоидную активацию](https://pytorch.org/docs/stable/generated/torch.nn.Sigmoid.html) после последнего линейного слоя сети. Таким образом мы гарантируем, что нейронная сеть предсказывает числа из промежутка $[0, 1]$. Использование активации - довольно распространенный прием, когда мы хотим получить числа из определенного диапазона значений.

In [ ]:
# 1.6 y в [0,1] + sigmoid на выходе

y_tr_n = normalize(y_train).astype(np.float32).reshape(-1, 1)
y_v_n = normalize(y_val).astype(np.float32).reshape(-1, 1)

# X пока без нормировки, только y
X_tr = torch.tensor(X_train, dtype=torch.float32)
X_v = torch.tensor(X_val, dtype=torch.float32)
y_tr = torch.tensor(y_tr_n)
y_v = torch.tensor(y_v_n)

train_loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=64, shuffle=True)
val_loader = DataLoader(TensorDataset(X_v, y_v), batch_size=64)


class NetSig(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(X_train.shape[1], 128)
        self.fc2 = nn.Linear(128, 1)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        return torch.sigmoid(self.fc2(x))  # чтобы выход был 0..1 как y


def mse_real(pred, y_n):
    # loss считается на норм y, а метрику хочу в годах
    p = denormalize(pred.detach().cpu().numpy()).reshape(-1)
    t = denormalize(y_n.detach().cpu().numpy()).reshape(-1)
    return mean_squared_error(t, p)


model = NetSig()
opt = torch.optim.SGD(model.parameters(), lr=1e-2)
crit = nn.MSELoss()

tr_mse, val_mse = train_and_validate(
    model, opt, crit, mse_real, train_loader, val_loader, num_epochs=4
)
print('norm y val mse:', round(val_mse, 2))

**Задание 1.7 (0.5 балла).** Сравните результаты этого эксперимента с предыдущим запуском.

**Ответ:** стало получше чем в 1.4, обучение спокойнее. Нормировка y + sigmoid помогли. До ridge еще не дотянули.

**Задание 1.8 (0.75 балла).** На этот раз попробуем отнормировать не только целевую переменную, но и сами данные, которые подаются сети на вход. Для них будем использовать нормализацию через среднее и стандартное отклонение. Преобразуйте данные и повторите прошлый эксперимент. Скорее всего, имеет смысл увеличить число эпох обучения.

In [ ]:
# 1.8 еще нормирую X - признаки же огромные по модулю

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)  # fit только train
X_val_s = scaler.transform(X_val)  # val тем же scaler

y_tr_n = normalize(y_train).astype(np.float32).reshape(-1, 1)
y_v_n = normalize(y_val).astype(np.float32).reshape(-1, 1)

X_tr = torch.tensor(X_train_s, dtype=torch.float32)
X_v = torch.tensor(X_val_s, dtype=torch.float32)
y_tr = torch.tensor(y_tr_n)
y_v = torch.tensor(y_v_n)

train_loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=64, shuffle=True)
val_loader = DataLoader(TensorDataset(X_v, y_v), batch_size=64)

model = NetSig()  # та же сеть что выше
opt = torch.optim.SGD(model.parameters(), lr=1e-2)
crit = nn.MSELoss()

# эпох побольше, иначе не успевает
tr_mse, val_mse = train_and_validate(
    model, opt, crit, mse_real, train_loader, val_loader, num_epochs=8
)
print('scaled X val mse:', round(val_mse, 2))
print('ridge for compare:', round(mse_ridge, 2))
# тут уже должно быть близко к ridge

Если вы все сделали правильно, то у вас должно было получиться качество, сравнимое с `Ridge` регрессией.

**Мораль:** как видите, нам пришлось сделать очень много хитрых телодвижений, чтобы нейронная сеть работала хотя бы так же, как и простая линейная модель. Здесь, конечно, показан совсем экстремальный случай, когда без нормализации данных нейронная сеть просто не учится. Как правило, в реальности завести нейронную сеть из коробки не очень сложно, но вот заставить ее работать на полную &mdash; куда более трудоемкая задача. Написание пайплайнов обучения нейросетевых моделей требует большой аккуратности, а дебаг часто превращается в угадайку. К счастью, очень часто на помощь приходит интуиция, и мы надеемся, что вы сможете выработать ее в течение нашего курса. Начнем с двух советов, которые стоит принять на вооружение:

- Обязательно начинаем любые эксперименты с бейзлайнов: без них мы бы не поняли, что нейронная сеть не учится в принципе.
- При постановке эксперментов старайтесь делать минимальное количество изменений за раз (в идеале одно!): только так можно понять, какие конкретно изменения влияют на результат.

## Часть 2. Улучшаем нейронную сеть (5 баллов)

Продолжим экспериментировать с нейронной сетью, чтобы добиться еще лучшего качества.

**Задание 2.1 (1 балл).** Давайте попробуем другие оптимизаторы. Обучите нейросеть с помощью SGD+momentum и Adam. Опишите свои наблюдения и в дальнейших запусках используйте лучший оптимизатор. Для Adam обычно берут learning rate поменьше, в районе $10^{-3}$.

In [ ]:
# 2.1 другие оптимизаторы
# беру пайплайн из 1.8 (scaled X + norm y)

def make_loaders():
    # каждый раз заново scaler - так проще, чем таскать глобально
    sc = StandardScaler()
    X_tr_s = sc.fit_transform(X_train)
    X_v_s = sc.transform(X_val)
    y_tr_n = normalize(y_train).astype(np.float32).reshape(-1, 1)
    y_v_n = normalize(y_val).astype(np.float32).reshape(-1, 1)

    tr = DataLoader(
        TensorDataset(torch.tensor(X_tr_s, dtype=torch.float32), torch.tensor(y_tr_n)),
        batch_size=64,
        shuffle=True,
    )
    vl = DataLoader(
        TensorDataset(torch.tensor(X_v_s, dtype=torch.float32), torch.tensor(y_v_n)),
        batch_size=64,
    )
    return tr, vl


def run_exp(opt_name, lr):
    tr, vl = make_loaders()
    model = NetSig()
    if opt_name == 'sgd':
        opt = torch.optim.SGD(model.parameters(), lr=lr)
    elif opt_name == 'sgd_mom':
        opt = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9)  # momentum 0.9 стандарт
    else:
        opt = torch.optim.Adam(model.parameters(), lr=lr)  # adam lr меньше

    _, val = train_and_validate(
        model, opt, nn.MSELoss(), mse_real, tr, vl, num_epochs=8, verbose=False
    )
    return val


res = {}
res['sgd'] = run_exp('sgd', 1e-2)
res['sgd_mom'] = run_exp('sgd_mom', 1e-2)
res['adam'] = run_exp('adam', 1e-3)  # для adam lr=1e-3 как в задании

print(res)
# adam норм, sgd без momentum хуже, дальше adam
best_opt = 'adam'

**Задание 2.2 (1 балл).** Теперь сделаем нашу нейронную сеть более сложной. Попробуйте сделать сеть:

- более широкой (то есть увеличить размерность скрытого слоя, например, вдвое)
- более глубокой (то есть добавить еще один скрытый слой)

Опишите, как увеличение числа параметров модели влияет на качество на обучающей и валидационной выборках.

In [ ]:
# 2.2 делаю сеть жирнее - посмотрим переобучится или нет

class NetWide(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(X_train.shape[1], 256)  # было 128, удвоил
        self.fc2 = nn.Linear(256, 1)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        return torch.sigmoid(self.fc2(x))


class NetDeep(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(X_train.shape[1], 128)
        self.fc2 = nn.Linear(128, 128)  # еще один скрытый
        self.fc3 = nn.Linear(128, 1)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        return torch.sigmoid(self.fc3(x))


def quick_train(net):
    # обертка чтобы не копипастить train_and_validate
    tr, vl = make_loaders()
    opt = torch.optim.Adam(net.parameters(), lr=1e-3)  # adam уже лучше
    tr_m, val_m = train_and_validate(
        net, opt, nn.MSELoss(), mse_real, tr, vl, num_epochs=8, verbose=False
    )
    return tr_m, val_m


tr_w, val_w = quick_train(NetWide())
tr_d, val_d = quick_train(NetDeep())
print('wide train/val:', round(tr_w, 2), round(val_w, 2))
print('deep train/val:', round(tr_d, 2), round(val_d, 2))
# на train лучше, на val хуже -> переобучение, как и ждали

**Задание 2.3 (1 балл).** Как вы должны были заметить, более сложная модель стала сильнее переобучаться. Попробуем добавить в обучение регуляризацию, чтобы бороться с переобучением. Добавьте слой дропаута ([`nn.Dropout`](https://pytorch.org/docs/stable/generated/torch.nn.Dropout.html#torch.nn.Dropout)) с параметром $p=0.2$ после каждого линейного слоя, кроме последнего. Почитать про дропаут можете в следующем [блогпосте](https://medium.com/@amarbudhiraja/https-medium-com-amarbudhiraja-learning-less-to-learn-better-dropout-in-deep-machine-learning-74334da4bfc5) или в оригинальной [статье](https://jmlr.org/papers/volume15/srivastava14a/srivastava14a.pdf)

Опишите результаты.

In [ ]:
# 2.3 dropout чтобы не зазубривала

class NetDrop(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(X_train.shape[1], 128)
        self.fc2 = nn.Linear(128, 128)
        self.fc3 = nn.Linear(128, 1)
        self.drop = nn.Dropout(0.2)  # p=0.2 из задания

    def forward(self, x):
        x = self.drop(torch.relu(self.fc1(x)))  # dropout после relu
        x = self.drop(torch.relu(self.fc2(x)))
        return torch.sigmoid(self.fc3(x))  # после последнего linear без dropout


tr_m, val_m = quick_train(NetDrop())
print('dropout train/val:', round(tr_m, 2), round(val_m, 2))
# val чуть лучше чем без dropout, train хуже - норм, так и должно быть

**Задание 2.4 (1.5 балла).** Теперь, когда мы определились с выбором архитектуры нейронной сети, пора заняться рутиной DL-инженера &mdash; перебором гиперпараметров. Подберите оптимальное значение lr по значению MSE на валидации (по логарифмической сетке, достаточно посмотреть 3-4 значения), можете воспользоваться `verbose=False` в функции `train_and_validate`.

Также подберем оптимальное значение параметра weight decay для данного lr. Weight decay &mdash; это аналог L2-регуляризации для нейронных сетей. Почитать о нем можно, например, [здесь](https://paperswithcode.com/method/weight-decay). В PyTorch он задается как параметр оптимизатора `weight_decay`. Подберите оптимальное значение weight decay по логарифимической сетке (его типичные значения лежат в диапазоне $[10^{-6}, 10^{-3}]$, но не забудьте включить нулевое значение в сетку).

Постройте графики зависимости MSE на трейне и на валидации от значений параметров. Прокомментируйте получившиеся зависимости.

In [ ]:
# 2.4 перебор lr и weight decay (долго...)

lrs = [1e-3, 1e-2, 1e-1]  # лог сетка как просят
lr_tr, lr_val = [], []

for lr in lrs:
    tr, vl = make_loaders()
    model = NetDrop()
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    tm, vm = train_and_validate(
        model, opt, nn.MSELoss(), mse_real, tr, vl, num_epochs=8, verbose=False
    )
    lr_tr.append(tm)
    lr_val.append(vm)

plt.figure(figsize=(7, 4))
plt.plot(lrs, lr_tr, 'o-', label='train')
plt.plot(lrs, lr_val, 'o-', label='val')
plt.xscale('log')
plt.xlabel('lr')
plt.ylabel('MSE')
plt.legend()
plt.show()

best_lr = lrs[np.argmin(lr_val)]  # мин val mse
print('best lr:', best_lr)

# теперь wd при лучшем lr
wds = [0, 1e-6, 1e-4, 1e-3]  # 0 тоже надо
wd_tr, wd_val = [], []

for wd in wds:
    tr, vl = make_loaders()
    model = NetDrop()
    opt = torch.optim.Adam(model.parameters(), lr=best_lr, weight_decay=wd)
    tm, vm = train_and_validate(
        model, opt, nn.MSELoss(), mse_real, tr, vl, num_epochs=8, verbose=False
    )
    wd_tr.append(tm)
    wd_val.append(vm)

plt.figure(figsize=(7, 4))
# log scale с нулем не работает, поэтому просто по индексам
plt.plot(range(len(wds)), wd_tr, 'o-', label='train')
plt.plot(range(len(wds)), wd_val, 'o-', label='val')
plt.xticks(range(len(wds)), wds)
plt.xlabel('weight decay')
plt.ylabel('MSE')
plt.legend()
plt.show()

best_wd = wds[np.argmin(wd_val)]
print('best wd:', best_wd)
# lr слишком большой -> val хуже, wd немного помогает от переобучения

Как вы могли заметить, еще одна рутина DL-инженера &mdash; утомительное ожидание обучения моделей.

**Задание 2.5 (0.5 балла).** Мы провели большое число экспериментов и подобрали оптимальную архитектуру и гиперпараметры. Пришло время обучить модель на полной обучающей выборке, померять качество на тестовой выборке и сравнить с бейзлайнами. Проделайте это.

In [ ]:
# 2.5 финал - учу на всем train, меряю test

# возвращаюсь к полному train (до val split)
X_full = X[:train_size]
y_full = y[:train_size]

# scaler заново на full train
scaler_full = StandardScaler()
X_full_s = scaler_full.fit_transform(X_full)
X_test_s = scaler_full.transform(X_test)  # test тем же scaler

# min/max тоже пересчитываю на full
y_min = y_full.min()
y_max = y_full.max()

y_full_n = normalize(y_full).astype(np.float32).reshape(-1, 1)
X_full_t = torch.tensor(X_full_s, dtype=torch.float32)
y_full_t = torch.tensor(y_full_n)

full_loader = DataLoader(TensorDataset(X_full_t, y_full_t), batch_size=64, shuffle=True)

model = NetDrop()  # лучшая архитектура из прошлых экспериментов
opt = torch.optim.Adam(model.parameters(), lr=best_lr, weight_decay=best_wd)

# val_loader тут не нужен, просто прогоню эпохи руками
for epoch in range(10):
    model.train()
    for X_b, y_b in full_loader:
        opt.zero_grad()
        pred = model(X_b)
        loss = nn.MSELoss()(pred, y_b)
        loss.backward()
        opt.step()

# test - denormalize обратно в годы
model.eval()
with torch.no_grad():
    pred_test = model(torch.tensor(X_test_s, dtype=torch.float32))
    pred_years = denormalize(pred_test.numpy()).reshape(-1)

mse_test = mean_squared_error(y_test, pred_years)
print('test mse nn:', round(mse_test, 2))
print('test mse ridge:', round(mse_ridge, 2))
print('test mse const:', round(mse_const, 2))
# сравниваю с бейзлайнами из задания 0